In [1]:
import pandas as pd
import numpy as np
from functools import reduce
from pathlib import Path
import glob
import os
import re

In [25]:
# requirement:
# all pathogenwatch outputs under the same directory
# checkm quality file place under pathogenwatch output directory
# a .tree file for corresponding pathogenwatch output
# a metadata files thats preprocessed

In [3]:
# import tables
# set pathogenwatch directory
#DIR = Path(r"C:/Users/Amoeb/Documents/ZSW2025-2026/Waterloo_courses/BIOL499A/nanopore/nanopore_racon_medaka/kraken_contig_result/Pathogenwatch/")
DIR = Path(r"C:/Users/Amoeb/Documents/ZSW2025-2026/Waterloo_courses/BIOL499A/nanopore/nanopore_medaka_polypolisher_pypolca/tree_output/")

CHECKM_file = "racon_medaka_quality_report.tsv"
OUTPUT_file = "ready_for_R_plot_final.csv"
# set regex for id/Genome Name
pattern = r'(.+?)_nanopore'

In [29]:
# import spn_pbp_amr, extract index, keep only needed columns
resistance = pd.read_csv(DIR / "spn_pbp_amr.csv")
resistance["Index"] = resistance["Genome Name"].str.extract(pattern)  # edit regex if needed

# readjust the pathogenwatch result by CLSI strandard Threshold
def extract_mic(val):
    if pd.isna(val):
        return np.nan
    val = str(val)
    match = re.search(r'[\d\.]+', val)
    return float(match.group()) if match else np.nan

mic_cols = ["amxMic", "croMic", "ctxMic", "memMic", "penMic"]
for col in mic_cols:
    resistance[col + "_num"] = resistance[col].apply(extract_mic)

# I typed it manually here, but actual data exist in MIC_TABLE_FINAL.xlsx
antibiotics = {
    "amxMic_num": {
        "name": "Amoxicillin",
        "bp": {"S": 2, "I": 4, "R": 8}},
    "croMic_num": {
        "name": "Ceftriaxone",
        "bp": {"S": 1, "I": 2, "R": 4}},
    "ctxMic_num": {
        "name": "Cefotaxime",
        "bp": {"S": 1, "I": 2, "R": 4}},
    "memMic_num": {
        "name": "Meropenem",
        "bp": {"S": 0.25, "I": 0.5, "R": 1}},
    "penMic_num": {
        "name": "Penicillin",
        "bp": {"S": 0.06, "I_low": 0.12, "I_high": 1, "R": 2}}
}

# sorry for my ugly code... Classify the S/I/R base of CLSI breakpoint
def classify_mic(val, bp):
    if pd.isna(val):
        return np.nan
    if "I_low" in bp:   # penicillin have upper and lower
        if val <= bp["S"]:
            return "S"
        elif bp["I_low"] <= val <= bp["I_high"]:
            return "I"
        elif val >= bp["R"]:
            return "R"
    else:
        if val <= bp["S"]:
            return "S"
        elif val == bp["I"]:
            return "I"
        elif val >= bp["R"]:
            return "R"
    return np.nan

for mic_col, info in antibiotics.items():
    antibiotic_name = info["name"]
    bp = info["bp"]
    new_col = antibiotic_name + "_adjusted"
    resistance[new_col] = resistance[mic_col].apply(
        lambda x: classify_mic(x, bp)
    )

cols_keep = [
    "Index",
    "Amoxicillin_adjusted",
    "Ceftriaxone_adjusted",
    "Cefotaxime_adjusted",
    "Meropenem_adjusted",
    "Penicillin_adjusted"
]

select_resist = resistance[cols_keep]

In [31]:
# import serotype, extract index, keep only needed columns
serotype = pd.read_csv(DIR / "serotype.csv")
serotype["Index"] = serotype["Genome Name"].str.extract(pattern)  # edit regex if needed
select_serotype = serotype[["Index", "Serotype"]]

In [33]:
# import mlst-pubmlst, extract index, keep only needed columns. 
# for ST that's longer than 10 string, reduce to 4, and add * in front
mlst = pd.read_csv(DIR / "mlst-pubmlst.csv")
mlst["Index"] = mlst["Genome Name"].str.extract(pattern)  # edit regex if needed
select_mlst = mlst[["Index", "ST"]]
mask = select_mlst["ST"].str.len() > 10
select_mlst.loc[mask, "ST"] = "*" + select_mlst.loc[mask, "ST"].str[:4]
select_mlst = select_mlst.rename(columns={"ST": "mlst"})

In [35]:
# a spot for genome completion from checkm or braken
checkm = pd.read_csv(DIR / CHECKM_file, delimiter='\t')
checkm["Index"] = checkm["Name"].str.extract(pattern)
selected_checkm = checkm[["Index", "Completeness"]]
selected_checkm = selected_checkm.rename(columns={"Completeness": "Genome_Completeness"})

In [37]:
selected_checkm

,Index,Genome_Completeness
0,i10_02_barcode11,99.99
1,i11_03_barcode74,98.65
2,i12_02_barcode35,100.00
3,i13_02_barcode17,99.99
4,i14_02_barcode41,100.00
5,i15_02_barcode09,99.99
6,i16_02_barcode57,100.00
7,i17_02_barcode51,100.00
8,i18_03_barcode91,86.20
9,i19_03_barcode20,91.21


In [39]:
# import metadata and add files label (using this same metadata here)
metadata = pd.read_csv("../nanopore/metadata_20260212.csv") # preprocessed manually
mapdata = pd.read_excel("../nanopore/illumina_nanopore_mapping.xlsx") # preprocessed with another script
indexed_metadata = mapdata.merge(metadata, on="accessionNum", how="left")

In [41]:
# merge all tables based on index
meta_pathogen_merged = (
    indexed_metadata
    .merge(select_resist,   on="Index", how="left")
    .merge(select_serotype, on="Index", how="left")
    .merge(select_mlst,     on="Index", how="left")
    .merge(selected_checkm, on="Index", how="left")
)

In [43]:
meta_pathogen_merged

,accessionNum,Index,Penicillin,Ceftriaxone,Erythromycin,Clindamycin,Vancomycin,Tetracycline,Levofloxacin,NP swap/ear swap,ageInMonths,Amoxicillin_adjusted,Ceftriaxone_adjusted,Cefotaxime_adjusted,Meropenem_adjusted,Penicillin_adjusted,Serotype,mlst,Genome_Completeness
0,W17143487,i2_01_barcode03,S,S,S,S,S,S,S,NP,31,S,S,S,S,S,23B,1262,99.99
1,F16380623,i3_01_barcode05,I,S,R,S,S,S,S,NP,30,I,S,S,I,R,35B,558,100.00
2,W17653277,i4_01_barcode11,I,S,R,S,S,S,S,NP,11,S,S,S,I,R,35B,*e0a6,99.99
3,M17701635,i5_01_04_barcode12_33,S,S,R,S,S,S,S,NP,18,I,S,S,I,R,35B,*dbf4,99.75
4,T18271326,i6_02_barcode90,S,S,S,S,S,S,S,NP,29,S,S,S,S,S,03,180,100.00
5,M18434558,i7_01_barcode34,S,S,S,S,S,S,S,NP,12,S,S,S,S,S,17F,392,100.00
6,F17654457,i8_01_barcode35,S,S,S,S,S,S,S,NP,6,S,S,S,S,S,17F,392,100.00
7,F17896493,i9_03_barcode89,S,S,S,S,S,S,S,NP,30,S,S,S,S,S,17F,*3daf,96.44
8,F18239423,i10_02_barcode11,I,S,R,S,S,S,S,NP,22,S,S,S,S,I,35B,156,99.99
9,T19376701,i11_03_barcode74,S,S,R,R,S,R,S,NP,6,NaN,NaN,NaN,NaN,NaN,03,*4f1d,98.65


In [45]:
# export merged file for R plotting
meta_pathogen_merged.to_csv(DIR / OUTPUT_file, index=False)

In [122]:
# after install biopython
# tree process for nanopore
from Bio import Phylo

tree = Phylo.read(DIR / "RAxML_bipartitions.spn_tree", "newick")

for leaf in tree.get_terminals():
    # rename reference
    if leaf.name == "GCF_003003495.1_ASM300349v1_genomic.fna.ref":
        leaf.name = "Reference"
        continue
        
    # set the leafs to the same as Index
    match = re.search(r"pattern", leaf.name)
    if match:
        leaf.name = match.group(1)

Phylo.write(tree, DIR / "RAxML_bipartitions_renamed.spn_tree", "newick")



1

In [15]:
# after install biopython
# tree process for illumina
from Bio import Phylo

tree = Phylo.read(DIR / "RAxML_bipartitions.spn_tree", "newick")

for leaf in tree.get_terminals():
    if leaf.name == "GCF_003003495.1_ASM300349v1_genomic.fna.ref":
        leaf.name = "reference"
    else:
        if leaf.name.startswith("snippy_"):
            leaf.name = leaf.name.removeprefix("snippy_")
        if leaf.name.endswith("_pypolca_corrected.fasta"):
            leaf.name = leaf.name.removesuffix("_pypolca_corrected.fasta")
Phylo.write(tree, DIR / "RAxML_bipartitions_renamed.spn_tree", "newick")


1

In [7]:
tree_ids = {leaf.name for leaf in tree.get_terminals()}
table_ids = set(indexed_metadata["Index"])

print("Missing in metadata:", tree_ids - table_ids)
print("Missing in tree:", table_ids - tree_ids)


NameError: name 'indexed_metadata' is not defined

In [ ]:
select_resist = resistance[["Index", "amx", "croNonMeningitis", "ctxNonMeningitis", "cxm", "mem", "penNonMeningitis", "penNonMeningitis_thresh"]]